In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# BlindDetection-V2：阶段2机制 canary

仅在控制会话审核后由用户手动运行。新生成4对图、216行：36倍率、12共享负样本、168对齐容差。不是阶段4或正式实验，science_denominator=0。当前无内容实测结果，未选定V2。


## 1. 加载已审阅代码
倍率0.5/0.75/1.0；角度0/-13/+7度。预设见 development.py 与 README.md。

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
EXACT = 'e602390e699c83b331c6e2c9ce60ead57b310112'
REPO = Path('/content/ceg-wm-v2-mechanism-e602390')
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V2/stage2-mechanism-v1')
if OUTPUT.exists():
    raise FileExistsError('保留已有运行目录，先检查结果，不覆盖或自动重跑。')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','BlindDetection-V2','--single-branch','https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('保留已有代码改动。')
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
print('Code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

## 2. 读取密钥与记录设备
开启Colab Secrets中的 HF_TOKEN 与 CEG_WM_ROOT_KEY 访问。现有生成器需要CUDA，不限定GPU型号或显存。

In [ ]:
from google.colab import userdata
import torch
for name in ('HF_TOKEN','CEG_WM_ROOT_KEY'):
    try:
        value=userdata.get(name)
    except Exception:
        raise RuntimeError(f'请开启Colab Secret {name} 的访问') from None
    if not value:
        raise RuntimeError(f'Empty secret: {name}')
    os.environ[name]=value
del value
print({'torch':torch.__version__,'cuda_available':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name() if torch.cuda.is_available() else None})

## 3. 运行216行

每单元生成一次content-only/primary-null对，共享一次SyncSeal残差。负样本跨倍率复用且只计一次。原生/生产warp评分同时覆盖正负样本。

21点容差在+7度、正样本0.75倍率下分别对正负两臂计算：零、角度±0.25/0.5/1/2度、水平/垂直各±1/2/4px，不做全乘积。真值仅用于诊断。

旧阈值仅描述参照；失败保留。初始化失败留下plan；不覆盖结果、不自动重跑。


In [ ]:
env=os.environ.copy()
env['PYTHONDONTWRITEBYTECODE']='1'
process=subprocess.run([sys.executable,str(REPO/'diagnostics/blind_detection_v2/development.py'),'--output',str(OUTPUT)],cwd=REPO,env=env)
print('Runner exit:',process.returncode)
if process.returncode:
    print('保留所有输出，检查错误和缺失行。')

## 4. 检查覆盖并交回控制会话
4对图仅机制canary，不能用于阶段4或正式校准/测试。审阅结果后再决定V2；这里不冻结方法、不正式运行。

In [ ]:
from collections import Counter
rows=[json.loads(line) for line in (OUTPUT/'rows.jsonl').read_text().splitlines()] if (OUTPUT/'rows.jsonl').exists() else []
def row_key(r):
    return tuple(r.get(k) for k in ('unit_id','kind','arm','strength','angle','angle_error','dx','dy'))
print({'planned_rows':216,'written_rows':len(rows),'unique_rows':len({row_key(r) for r in rows}),'row_errors':sum(r.get('error') is not None for r in rows),'kinds':dict(Counter(r['kind'] for r in rows))})
if (OUTPUT/'summary.json').exists():
    print((OUTPUT/'summary.json').read_text())
else:
    print('未完成：无终态summary。保留已有文件与plan。')